In [ ]:
# google colabの場合実行
# !uv pip install torch_geometric==2.8.1

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch_geometric.datasets import Amazon
from torch_geometric.nn import GCNConv
from torch_geometric.utils import degree, to_networkx, homophily


In [ ]:
seed = 42
y_col = "product_category"

## 1. データの読み込み

In [ ]:
dataset = Amazon(root="./data/Amazon", name="Computers")


In [ ]:
data = dataset[0]

In [ ]:
data

In [ ]:
df_features = pd.DataFrame(data.x.detach().cpu().numpy())

In [ ]:
# カラム名の設定
# 商品をnodeとみなしている
df_features.columns = [f"feat_{i}" for i in range(data.num_node_features)]
df_features.index = [f"node_{i}" for i in range(data.num_nodes)]

In [ ]:
df_features.head(3)

In [ ]:
# 目的変数の読み込み
df_features[y_col] = data.y.detach().cpu().numpy()

In [ ]:
df_features.head(3)

## 2. データの中身の確認

In [ ]:
# データセット全体の規模の確認
print(f"ノード数: {data.num_nodes}")
print(f"エッジ数: {data.num_edges}")
print(f"特徴量数: {data.num_node_features}")
print(f"クラス数: {dataset.num_classes}")

### 2.1 データ型などの確認

In [ ]:
df_features.shape

In [ ]:
df_features.dtypes

In [ ]:
# 欠損値の確認
df_features.isnull().sum().sum()

# →欠損値はなし

In [ ]:
# binary以外の特徴量カラムの確認

is_binary = np.isin(
      df_features.drop(columns=[y_col]).to_numpy(), [0, 1]
    ).all()

print(f"すべての特徴量がバイナリか: {is_binary}")

# →特徴量はすべてbinaryデータ

### 2.2 グラフネットワークとしての特徴の確認

In [ ]:
data.is_undirected()

# →trueなので、無向グラフ。一方通行の矢印ではなく、双方向のグラフ。

In [ ]:
# 次数の確認
# GNNのメッセージパッシングが次数の影響を強く受けるため

node_degrees = degree(data.edge_index[0], num_nodes=data.num_nodes)

In [ ]:
print(f"最大次数: {node_degrees.max().item()}")
print(f"平均次数: {node_degrees.mean().item():.2f}")
print(f"孤立ノード数: {(node_degrees == 0).sum().item()}")

In [ ]:
# 次数分布のヒストグラム

plt.hist(node_degrees.numpy(), bins=50, log=True)
plt.xlabel("Degree")
plt.ylabel("Count (Log Scale)")
plt.title("Node Degree Distribution")
plt.show()

In [ ]:
# NetworkX形式に変換

G = to_networkx(data, to_undirected=True)

In [ ]:
print(f"グラフの密度: {nx.density(G):.5f}")
print(f"連結成分の数（島の数）: {nx.number_connected_components(G)}")

In [ ]:
# 同質性（Homophily）の計測
# 「同じクラスのノード同士がリンクで結ばれやすいか、を確認

h = homophily(data.edge_index, data.y, method="edge")
print(f"エッジ同質性 (Homophily Ratio): {h:.4f}")

# →エッジ同質性が約0.777であり、同じクラスのノード同士がつながる傾向が見られる。グラフ構造を利用するGNNが有効である可能性がある。

### 2.3 ノードを次元圧縮して、クラスごとに描画

In [ ]:
X = df_features.drop(y_col, axis=1)
y = df_features[y_col]

In [ ]:
# t-SNEのモデル作成と実行
# 高次元のデータを低次元に圧縮し、「元の空間で近いデータ同士を、低い次元でも近くに配置する」ための可視化手法
# グラフ構造なしで分類できそうかを簡易的に確認
tsne = TSNE(n_components=2, perplexity=30, random_state=seed)
X_embedded = tsne.fit_transform(X)

In [ ]:
# 描画
plt.scatter(
    X_embedded[:, 0],
    X_embedded[:, 1],
    c=y,
    cmap="hsv",
    alpha=0.6
)

plt.colorbar()
plt.show()

# →特徴量を圧縮しているという前提はあるものの、グラフ構造なしだと、図の上ではクラスが混ざっている。

### 2.4 目的変数の確認

In [ ]:
# 目的変数の分布を確認

df_features[y_col].value_counts().sort_index().plot.bar()